# 03 Visualisierung – Data Understanding

- **Teil 1 – Datenqualität:** Arbeitsteil, liefert die Maßnahmen für die Data Preparation.
- **Teil 2 – Explorative Analyse:** wird in der Präsentation gezeigt (nur Grafiken + Kernaussagen).

Datenbasis: `../data/processed/credit_dtypes_pcd.parquet` (erzeugt in `02_Datatypes`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns
from sklearn.metrics import roc_auc_score

# --- Einheitliches Design ---------------------------------------------------
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titlesize": 16, "figure.titlesize": 16,
    "axes.labelsize": 13,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "patch.force_edgecolor": True,   # feine Rahmen, damit Balken nicht verschmelzen
    "patch.edgecolor": "white",
    "patch.linewidth": 0.6,
})

FARBE_ZAHLT = "#4C72B0"    # zahlt zurück
FARBE_AUSFALL = "#C44E52"  # Ausfall
FARBE_NEUTRAL = "#8C8C8C"  # Referenzlinien
FARBE_HAUPT = "#4C72B0"    # einfarbige Verteilungsgrafiken

GROESSE_EINZEL = (9, 5)
GROESSE_RASTER = (14, 9)
GROESSE_REIHE = (16, 5)    # drei Grafiken nebeneinander
FLIERPROPS = {"alpha": 0.2, "markersize": 3}

JA_NEIN = {False: "Nein", True: "Ja"}


# --- Hilfsfunktionen ----------------------------------------------------------
def tausender(ax, achse="y"):
    """Achse mit deutschem Tausenderpunkt formatieren (8000 -> 8.000)."""
    fmt = FuncFormatter(lambda x, _: f"{x:,.0f}".replace(",", "."))
    (ax.yaxis if achse == "y" else ax.xaxis).set_major_formatter(fmt)


def balken_haeufigkeit(ax, serie, titel, xlabel):
    """Balkendiagramm der Häufigkeiten einer kategorischen Spalte."""
    counts = serie.value_counts()
    ax.bar(counts.index.astype(str), counts.values, color=FARBE_HAUPT)
    ax.set_title(titel)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Amount of Loans")
    ax.tick_params(axis="x", rotation=30)
    for lbl in ax.get_xticklabels():
        lbl.set_horizontalalignment("right")
    tausender(ax)


def balken_ausfallquote(ax, daten, spalte, titel, gesamtquote, sortieren=True):
    """Ausfallquote je Gruppe mit gestrichelter Gesamtlinie (absteigend sortiert, außer bei Ordinalgruppen)."""
    q = daten.groupby(spalte, observed=True)["loan_status"].mean().mul(100)
    if sortieren:
        q = q.sort_values(ascending=False)
    ax.bar(q.index.astype(str), q.values, color=FARBE_AUSFALL)
    ax.axhline(gesamtquote, color=FARBE_NEUTRAL, linestyle="--",
               label=f"Gesamt: {gesamtquote:.1f} %".replace(".", ","))
    ax.set_title(f"nach {titel}")
    ax.set_xlabel(titel)
    ax.set_ylabel("Ausfallquote in %")
    ax.tick_params(axis="x", rotation=30)
    for lbl in ax.get_xticklabels():
        lbl.set_horizontalalignment("right")
    ax.set_ylim(0, q.max() * 1.25)  # Platz für die Legende
    ax.legend(loc="upper right")

In [ ]:
df_dtypes_pcd = pd.read_parquet("../data/processed/credit_dtypes_pcd.parquet")
display(df_dtypes_pcd.head(5))

---
# Teil 1: Datenqualität

### 1.1 Überblick

In [ ]:
print("Zeilen x Spalten:", df_dtypes_pcd.shape)
print("Komplett doppelte Zeilen:", df_dtypes_pcd.duplicated().sum())
display(df_dtypes_pcd.dtypes.to_frame("Datentyp"))

### 1.2 Fehlende Werte und Nullen
Eine 0 ist kein fehlender Wert – aber man sollte sie von NaN unterscheiden
(z. B. 0 Jahre Beschäftigung ist plausibel, ein Zinssatz von 0 % nicht).

In [ ]:
n = len(df_dtypes_pcd)
num = df_dtypes_pcd.select_dtypes(include="number")   # ohne boolean/category

fehlend = pd.DataFrame({
    "NaN Anzahl": df_dtypes_pcd.isna().sum(),
    "NaN Anteil %": (df_dtypes_pcd.isna().mean() * 100).round(2),
    "exakt 0 Anzahl": (num == 0).sum(),
    "exakt 0 Anteil %": ((num == 0).mean() * 100).round(2),
})
display(fehlend)

**Befund:** Nur zwei Spalten haben fehlende Werte: `loan_int_rate` (3.116 NaN, 9,56 %) und
`person_emp_length` (895 NaN, 2,75 %). `loan_int_rate` hat **keine** Nullen – die Lücken sind echte NaN.
`person_emp_length` ist zusätzlich 4.105-mal exakt 0 (12,6 %), das ist ein gültiger Wert.

### 1.3 Wer hat fehlende Werte?

In [ ]:
def profil(gruppen):
    return df_dtypes_pcd.groupby(gruppen).agg(
        Anzahl=("person_age", "size"),
        Median_Alter=("person_age", "median"),
        Median_Einkommen=("person_income", "median"),
        Ausfallquote_pct=("loan_status", lambda s: round(s.mean() * 100, 1)),
    )

# a) Zinssatz vorhanden vs. fehlt
zins = df_dtypes_pcd["loan_int_rate"].isna().map({False: "Zins vorhanden", True: "Zins fehlt"})
display(profil(zins.rename("Zinssatz")))

# b) Beschäftigungsdauer: fehlt / 0 Jahre / > 0 Jahre
emp = pd.Series("> 0 Jahre", index=df_dtypes_pcd.index, name="Beschäftigungsdauer")
emp[df_dtypes_pcd["person_emp_length"] == 0] = "0 Jahre"
emp[df_dtypes_pcd["person_emp_length"].isna()] = "fehlt"
display(profil(emp).loc[["fehlt", "0 Jahre", "> 0 Jahre"]])

**Zinssatz:** Personen mit und ohne Zinsangabe unterscheiden sich kaum (Median-Alter 26 vs. 26,
Median-Einkommen 55.000 vs. 55.640, Ausfallquote 21,9 % vs. 20,7 %). Der Zins fehlt offenbar **zufällig**.

**Beschäftigungsdauer:** Sie fehlt **nicht** bei besonders Jungen (Median-Alter 25 statt 26), sondern bei
**Geringverdienern** (Median-Einkommen 36.000 statt 57.200) mit **höherer Ausfallquote** (31,5 % statt 20,6 %).
Das Fehlen ist also selbst eine Information → beim Auffüllen eine Indikatorspalte `emp_fehlt` anlegen.

### 1.4 Ausreißer in den numerischen Merkmalen
Ein Boxplot je numerischem Merkmal; Punkte außerhalb der Whisker gelten als Ausreißer (IQR-Regel).
Besonders wichtig sind **unrealistische** Werte, z. B. ein Alter über 100 Jahre oder eine
Beschäftigungsdauer, die länger ist als das Erwerbsleben.

In [ ]:
num_cols = {
    "person_age": "Alter (Jahre)",
    "person_income": "Einkommen (log. Skala)",
    "person_emp_length": "Beschäftigung (Jahre)",
    "loan_amnt": "Kredithöhe",
    "cb_person_cred_hist_length": "Kredithistorie (Jahre)",
}

fig, axes = plt.subplots(1, len(num_cols), figsize=(18, 5))
for ax, (col, titel) in zip(axes, num_cols.items()):
    sns.boxplot(y=df_dtypes_pcd[col], ax=ax, color=FARBE_HAUPT, flierprops=FLIERPROPS)
    ax.set_title(titel, fontsize=13)
    ax.set_ylabel("")
    if col == "person_income":
        ax.set_yscale("log")          # sonst drückt der 6-Mio.-Wert die Box platt
    else:
        tausender(ax)
fig.suptitle("Abb. 1: Ausreißer in den numerischen Merkmalen")
plt.tight_layout()
plt.show()

# --- Ausreißer-Übersicht (IQR-Regel, wie im Boxplot) ---
rows = []
for col in num_cols:
    s = df_dtypes_pcd[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((s < lower) | (s > upper)).sum()
    rows.append({"Merkmal": col, "Untere Grenze": lower, "Obere Grenze": upper,
                 "Min": s.min(), "Max": s.max(), "Anzahl Ausreißer": n_out,
                 "Anteil %": round(n_out / len(s) * 100, 2)})
display(pd.DataFrame(rows).set_index("Merkmal").round(1))

# Die extremsten bzw. unplausiblen Fälle
display(df_dtypes_pcd.nlargest(5, "person_age")[["person_age", "person_income", "person_emp_length"]])
unplausibel = df_dtypes_pcd["person_emp_length"] > df_dtypes_pcd["person_age"] - 14
display(df_dtypes_pcd.loc[unplausibel, ["person_age", "person_emp_length"]])
print("Alter > 100:", (df_dtypes_pcd["person_age"] > 100).sum(),
      "| Beschäftigung > Alter - 14:", unplausibel.sum())

**Befund:** 5 Personen sind älter als 100 Jahre (bis 144) und 2 Personen arbeiten angeblich länger als
Alter − 14 Jahre (123 Jahre Beschäftigung) – das sind Eingabefehler. Die vielen Einkommens-Ausreißer
nach oben (bis 6 Mio.) sind dagegen plausibel (Spitzenverdiener) und bleiben im Datensatz.

### 1.6 Fazit Teil 1: Befund → Maßnahme in der Data Preparation

| Befund | Maßnahme in der Data Preparation |
|---|---|
| 165 komplett doppelte Zeilen | Duplikate nicht entfernen |
| 5 Personen älter als 100 Jahre | Zeilen löschen (Eingabefehler) |
| 2 Personen mit Beschäftigung > Alter − 14 | Zeilen löschen (Eingabefehler) |
| `person_emp_length` fehlt bei 895 Personen (2,75 %), v. a. Geringverdiener mit höherer Ausfallquote | Mit dem Median füllen (nur aus Trainingsdaten) + Indikatorspalte `emp_fehlt` |
| `loan_int_rate` fehlt bei 3.116 Personen (9,56 %), offenbar zufällig | Spalte wird ohnehin entfernt (Data Leakage, siehe Teil 2.4) |
| Sehr hohe Einkommen (bis 6 Mio.) | Behalten – plausibel, keine Fehler |
| Unausgewogene Zielvariable (78 : 22) | `stratify=y` beim Train-Test-Split |

---
# Teil 2: Explorative Analyse

---
### 2.1 Verteilung der Zielvariable: Fehlerquote der manuellen Prüfung

Der Datensatz enthält nur **genehmigte** Kredite. Jeder Kredit wurde also von einem Mitarbeiter als „zahlt zurück“ eingeschätzt.
Mit **Ausfall (1) als positiver Klasse** gilt deshalb:

- **True Negatives:** genehmigt und zurückgezahlt (`loan_status = 0`)
- **False Negatives:** genehmigt, aber ausgefallen (`loan_status = 1`)

Der Anteil der Ausfälle ist damit die **False-Negative-Quote der manuellen Prüfung**, also der Anteil der Fehlentscheidungen unter den genehmigten Krediten.

**Einschränkung:** False Positives, also abgelehnte Antragsteller, die zurückgezahlt hätten, sind im Datensatz nicht enthalten (Survivorship Bias). Die vollständige Fehlerquote der Mitarbeiter lässt sich deshalb nicht berechnen, nur ihre Fehler bei den genehmigten Krediten.

In [ ]:
counts = df_dtypes_pcd["loan_status"].value_counts().sort_index()
percent = df_dtypes_pcd["loan_status"].value_counts(normalize=True).sort_index() * 100

fig, ax = plt.subplots(figsize=GROESSE_EINZEL)
bars = ax.bar(["zahlt zurück (0)", "Ausfall (1)"], counts.values,
              color=[FARBE_ZAHLT, FARBE_AUSFALL])
ax.bar_label(bars, padding=3, fontsize=12,
             labels=[f"{c:,}".replace(",", ".") + f" ({p:.1f} %)".replace(".", ",")
                     for c, p in zip(counts.values, percent.values)])
ax.set_title("Abb. 2: Verteilung des Kreditstatus")
ax.set_xlabel("Kreditstatus")
ax.set_ylabel("Anzahl Kredite")
ax.set_ylim(0, counts.max() * 1.15)  # Platz für die Beschriftung
tausender(ax)
plt.tight_layout()
plt.show()

**Befund:** 21,8 % der Kredite fallen aus – deutlich mehr als die 7 % aus der Aufgabenstellung.
Die Klassen sind unausgewogen (ca. 78 : 22), daher muss beim Train-Test-Split stratifiziert werden.

---
### 2.2 Kreditvolumen at Risk

Wie viel Kreditvolumen steckt in den ausgefallenen Krediten, verglichen mit dem Gesamtvolumen?
`loan_amnt` ist der ausgezahlte Betrag. Der tatsächliche Verlust kann durch Rückflüsse niedriger sein.

In [ ]:
ausfall = df_dtypes_pcd['loan_status'].astype(bool)
vol_gesamt  = df_dtypes_pcd['loan_amnt'].sum()
vol_ausfall = df_dtypes_pcd.loc[ausfall, 'loan_amnt'].sum()

print(f"Volumen at Risk: {vol_ausfall:,.0f} von {vol_gesamt:,.0f} ({vol_ausfall / vol_gesamt:.1%})")

werte = [vol_gesamt / 1e6, vol_ausfall / 1e6]
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(['Gesamtvolumen', 'Volumen at Risk'], werte, color=['grey', 'red'])
ax.bar_label(bars, labels=[f'{werte[0]:,.1f} Mio.', f'{werte[1]:,.1f} Mio. ({vol_ausfall / vol_gesamt:.1%})'],
             padding=3)
ax.set_title('Kreditvolumen at Risk durch Zahlungsausfälle')
ax.set_ylabel('Volumen in Mio.')
ax.set_ylim(0, werte[0] * 1.15)
plt.show()

### 2.3 Wer leiht sich Geld?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=GROESSE_REIHE)

axes[0].hist(df_dtypes_pcd["person_income"] / 1000, bins=50, range=(0, 300), color=FARBE_HAUPT)
axes[0].set_title("Einkommen (bis 300 Tsd.)")
axes[0].set_xlabel("Jahreseinkommen in Tsd.")

axes[1].hist(df_dtypes_pcd["person_age"], bins=40, range=(20, 60), color=FARBE_HAUPT)
axes[1].set_title("Alter (bis 60 Jahre)")
axes[1].set_xlabel("Alter in Jahren")

axes[2].hist(df_dtypes_pcd["loan_amnt"] / 1000, bins=30, color=FARBE_HAUPT)
axes[2].set_title("Kredithöhe")
axes[2].set_xlabel("Kredithöhe in Tsd.")

for ax in axes:
    ax.set_ylabel("Anzahl Kredite")
    tausender(ax)
fig.suptitle("Abb. 3: Einkommen, Alter und Kredithöhe der Kreditnehmer")
plt.tight_layout()
plt.show()

**Typischer Kreditnehmer (Median):**
Der Median wird statt des Mittelwerts verwendet, da Alter und Einkommen starke Ausreißer enthalten.

In [ ]:
profil = pd.Series({
    'Alter (Jahre)':               df_dtypes_pcd['person_age'].median(),
    'Einkommen':                   df_dtypes_pcd['person_income'].median(),
    'Beschäftigungsdauer (Jahre)': df_dtypes_pcd['person_emp_length'].median(),
    'Kredithöhe':                  df_dtypes_pcd['loan_amnt'].median(),
    'Verwendungszweck (Modus)':    df_dtypes_pcd['loan_intent'].mode()[0],
    'Wohnsituation (Modus)':       df_dtypes_pcd['person_home_ownership'].mode()[0],
})
display(profil.to_frame('Typischer Wert'))

### Erkenntnis
Die Kreditnehmer sind jung (Median 26 Jahre) mit mittlerem Einkommen (Median 55.000), der typische
Kredit ist klein (Median 8.000) → es handelt sich überwiegend um Konsumentenkredite, nicht um Hypotheken.

### 2.4 Wofür und von wem?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=GROESSE_REIHE)
balken_haeufigkeit(axes[0], df_dtypes_pcd["loan_intent"], "Loan Purpose", "Purpose")
balken_haeufigkeit(axes[1], df_dtypes_pcd["person_home_ownership"], "Wohnsituation", "Wohnsituation")
balken_haeufigkeit(axes[2], df_dtypes_pcd["cb_person_default_on_file"].map(JA_NEIN),
                   "Früherer Kreditausfall", "Früher ausgefallen?")
fig.suptitle("Abb. 4: Verwendungszweck, Wohnsituation und früherer Ausfall")
plt.tight_layout()
plt.show()

**Die Verwendungszwecke sind recht gleichmäßig verteilt (Bildung am häufigsten mit 19,8 %), die meisten
Kreditnehmer wohnen zur Miete (50,5 %) und 17,6 % sind schon einmal mit einem Kredit ausgefallen.**

### 2.5 Welche Gruppen fallen häufiger aus?

In [ ]:
df_plot = df_dtypes_pcd.copy()
df_plot["altersgruppe"] = pd.cut(df_plot["person_age"], bins=[0, 25, 30, 40, 50, 200],
                                 labels=["bis 25", "26–30", "31–40", "41–50", "über 50"])
df_plot["frueherer_ausfall"] = df_plot["cb_person_default_on_file"].map(JA_NEIN)
gesamtquote = df_plot["loan_status"].mean() * 100

merkmale = {
    "loan_intent": "Verwendungszweck",
    "person_home_ownership": "Wohnsituation",
    "altersgruppe": "Altersgruppe",
    "frueherer_ausfall": "früherem Ausfall",
}

fig, axes = plt.subplots(2, 2, figsize=GROESSE_RASTER)
for ax, (spalte, titel) in zip(axes.flat, merkmale.items()):
    balken_ausfallquote(ax, df_plot, spalte, titel, gesamtquote, sortieren=(spalte != "altersgruppe"))
axes[1, 1].set_xlabel("Früher ausgefallen?")
fig.suptitle("Abb. 5: Ausfallquote je Gruppe")
plt.tight_layout()
plt.show()

### Erkenntnis:
Mieter fallen mit 31,6 % gut viermal so oft aus wie Eigentümer (7,5 %), und wer schon einmal ausgefallen
ist, fällt mit 37,8 % doppelt so oft aus (sonst 18,4 %). Das Alter spielt dagegen kaum eine Rolle (20–25 % in allen Gruppen).

### 2.6 Welches Merkmal hängt am stärksten mit dem Ausfall zusammen?

In [ ]:
df_corr = df_dtypes_pcd.copy()
bool_cols = ["loan_status", "cb_person_default_on_file"]
df_corr[bool_cols] = df_corr[bool_cols].astype(float)
df_corr = pd.get_dummies(df_corr, columns=["person_home_ownership", "loan_intent"], dtype=float)

corr = df_corr.corr(method="spearman")
ziel = corr[["loan_status"]].drop("loan_status").sort_values("loan_status", ascending=False)

fig, ax = plt.subplots(figsize=(7, 9))
sns.heatmap(ziel, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, center=0,
            linewidths=0.5, linecolor="white", ax=ax, annot_kws={"size": 11},
            cbar_kws={"label": "Spearman-Korrelation r"})
ax.set_title("Abb. 6: Korrelation mit Zahlungsausfall")
ax.set_xlabel("")
ax.set_ylabel("Merkmal")
ax.set_xticklabels(["Ausfall (loan_status)"])
plt.tight_layout()
plt.show()

Der Zinssatz hat mit r = 0,32 die stärkste Korrelation mit dem Ausfall, gefolgt von Mieten (0,24);
höheres Einkommen geht mit weniger Ausfällen einher (r = −0,27). Alle Zusammenhänge sind nur schwach bis mäßig.

### 2.7 Ist der Zins ein Ergebnis der Risikoprüfung?

In [ ]:
fig, ax = plt.subplots(figsize=GROESSE_EINZEL)
sns.boxplot(data=df_plot, x="frueherer_ausfall", y="loan_int_rate", order=["Nein", "Ja"],
            hue="frueherer_ausfall", palette={"Nein": FARBE_ZAHLT, "Ja": FARBE_AUSFALL},
            legend=False, flierprops=FLIERPROPS, ax=ax)

mediane = df_plot.groupby("frueherer_ausfall")["loan_int_rate"].median()
for i, gruppe in enumerate(["Nein", "Ja"]):
    ax.text(i, mediane[gruppe], f"{mediane[gruppe]:.1f} %".replace(".", ","),
            ha="center", va="bottom", fontsize=11, fontweight="bold", color="white",
            bbox=dict(facecolor="black", alpha=0.6, pad=2, edgecolor="none"))

ax.set_title("Abb. 7: Zinssatz nach früherem Kreditausfall")
ax.set_xlabel("Früher schon einmal ausgefallen?")
ax.set_ylabel("Zinssatz in %")
plt.tight_layout()
plt.show()

m = df_dtypes_pcd["loan_int_rate"].notna()
auc_zins = roc_auc_score(df_dtypes_pcd.loc[m, "loan_status"].astype(int), df_dtypes_pcd.loc[m, "loan_int_rate"])
print(f"ROC-AUC nur mit dem Zinssatz (ohne NaN): {auc_zins:.3f}")

### Erkenntnis
Wer früher schon ausgefallen ist, bekommt deutlich höhere Zinsen (Median 14,3 % statt 10,4 %).
Der Zinssatz allein trennt Ausfall und Rückzahlung bereits mit AUC = 0,72 → er enthält schon die
Risikoeinschätzung der Bank.

### 2.7 Fazit
Der Zinssatz wird von der Bank **nach** ihrer Risikoprüfung festgelegt. Ein Modell, das `loan_int_rate`
nutzt, würde also die Entscheidung der bisherigen manuellen Prüfung mitlernen, statt selbst Risiko zu
bewerten – bei einem neuen Antrag liegt der Zins zum Zeitpunkt der Entscheidung noch gar nicht vor
(**Data Leakage**). Deshalb wird `loan_int_rate` vor dem Training entfernt. Wie stark sich das auf die
Modellgüte auswirkt, zeigen wir in `05_Modell_Tuning` im Vergleich mit/ohne Zins.